# 02 — Web Scraping (Wikipedia Circuit Pages)

**Project:** F1 Track Personality 2020–2025  
**Authors:** Matteo Asscher & Aksel  
**Goal:** Scrape circuit-level metadata (length, corners, type, location) from Wikipedia for every circuit appearing in the 2020–2025 F1 seasons. This enriches the jolpica API data from `01_api_collection.ipynb` so we can analyze whether constructor over/under-performance correlates with track characteristics.

**Source:** English Wikipedia — individual circuit pages, e.g. `https://en.wikipedia.org/wiki/Bahrain_International_Circuit`. Linked from each `Circuit.url` field returned by the jolpica API.

---

## Ethics & Source Note

- **Source URL:** `https://en.wikipedia.org/` (one page per F1 circuit, ~30 pages total)
- **robots.txt verdict:** verified programmatically in Cells 2–5 below. Wikipedia's robots.txt explicitly welcomes "friendly, low-speed bots viewing article pages." Our User-Agent is not on any of the named blocklists. Article URLs under `/wiki/` are confirmed allowed.
- **License:** Wikipedia article text is published under CC BY-SA 4.0. Data we extract (track length, year built, etc.) is factual information not subject to copyright; we will cite Wikipedia as the source in the final report.
- **Rate limit applied:** `time.sleep(1 + random.random())` between requests (~1.5s on average). Wikipedia has no published rate limit but recommends a polite User-Agent and not exceeding a few requests per second.
- **User-Agent:** identifies us as a student project with a contact reference, per Wikipedia's API etiquette guidelines.
- **Personal data audit:** none of the scraped fields contain personal data. Circuit pages describe physical infrastructure (length, location, surface). Driver names appear on these pages but we are not extracting them. GDPR does not apply.

**Status (last run):** _will be updated after the scraping completes — number of circuits collected, fields populated, and any gaps._

In [1]:
import json
import random
import time
from pathlib import Path
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

# User-Agent that identifies us politely to Wikipedia (see Ethics & Source Note above)
USER_AGENT = "AlbertSchool-F1Project/1.0 (matteo.asscher@me.com; educational use)"

# Standard headers we'll attach to every Wikipedia request
HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Language": "en-US,en;q=0.9",
}

# Paths to data folders
DATA_RAW = Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Setup OK")
print(f"  pandas:   {pd.__version__}")
print(f"  requests: {requests.__version__}")
print(f"  bs4:      installed")

Setup OK
  pandas:   3.0.3
  requests: 2.34.2
  bs4:      installed


In [2]:
# Fetch robots.txt with our identifying User-Agent, then parse the rules.
# (RobotFileParser fetches with no User-Agent by default, which Wikipedia blocks with 403.)

resp = requests.get(
    "https://en.wikipedia.org/robots.txt",
    headers=HEADERS,
    timeout=10,
)
resp.raise_for_status()

rp = RobotFileParser()
rp.parse(resp.text.splitlines())

# Test a representative set of paths: two F1 circuit articles (our actual targets),
# plus Special:Random and the api endpoint (we expect these to be disallowed).
test_urls = [
    "https://en.wikipedia.org/wiki/Bahrain_International_Circuit",
    "https://en.wikipedia.org/wiki/Circuit_de_Monaco",
    "https://en.wikipedia.org/wiki/Special:Random",
    "https://en.wikipedia.org/w/api.php",
]

print(f"robots.txt status: {resp.status_code}")
print(f"User-Agent in use: {USER_AGENT}\n")
print(f"{'URL':<65} {'Allowed?'}")
print("-" * 80)
for url in test_urls:
    allowed = rp.can_fetch(USER_AGENT, url)
    print(f"{url:<65} {'✅ yes' if allowed else '❌ no'}")

robots.txt status: 200
User-Agent in use: AlbertSchool-F1Project/1.0 (matteo.asscher@me.com; educational use)

URL                                                               Allowed?
--------------------------------------------------------------------------------
https://en.wikipedia.org/wiki/Bahrain_International_Circuit       ✅ yes
https://en.wikipedia.org/wiki/Circuit_de_Monaco                   ✅ yes
https://en.wikipedia.org/wiki/Special:Random                      ❌ no
https://en.wikipedia.org/w/api.php                                ❌ no


In [3]:
# Fetch a single circuit page to inspect its structure before scraping all of them
url = "https://en.wikipedia.org/wiki/Bahrain_International_Circuit"

resp = requests.get(url, headers=HEADERS, timeout=10)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "lxml")

print(f"Status: {resp.status_code}")
print(f"Page title: {soup.title.get_text(strip=True)}")
print(f"HTML length: {len(resp.text):,} characters")

Status: 200
Page title: Bahrain International Circuit - Wikipedia
HTML length: 308,795 characters


In [4]:
# Find the infobox (the panel on the right side of any Wikipedia article)
infobox = soup.find("table", class_="infobox")

if infobox is None:
    print("❌ No infobox found on this page")
else:
    print("✅ Found infobox")
    print(f"Number of rows in infobox: {len(infobox.find_all('tr'))}\n")
    
    # Print the first 15 rows to see the structure: label on the left, value on the right
    print("First 15 rows of the infobox:")
    print("-" * 80)
    for row in infobox.find_all("tr")[:15]:
        label = row.find("th")
        value = row.find("td")
        label_text = label.get_text(" ", strip=True) if label else "(no label)"
        value_text = value.get_text(" ", strip=True) if value else "(no value)"
        print(f"  {label_text!r:<35} → {value_text[:70]!r}")

✅ Found infobox
Number of rows in infobox: 36

First 15 rows of the infobox:
--------------------------------------------------------------------------------
  '(no label)'                        → ''
  '(no label)'                        → 'Grand Prix Circuit (2005–present)'
  'Location'                          → 'Sakhir , Bahrain'
  'Coordinates'                       → '26°1′57″N 50°30′38″E \ufeff / \ufeff 26.03250°N 50.51056°E \ufeff / 26.03250; 50.5105'
  'Capacity'                          → '70,000'
  'FIA Grade'                         → '1 (5 layouts)'
  'Broke ground'                      → 'December\xa02002 ; 23\xa0years ago ( 2002-12 )'
  'Opened'                            → '17\xa0March 2004 ; 22 years ago ( 2004-03-17 )'
  'Construction cost'                 → '56.2 million Dinars ($150 million)'
  'Architect'                         → 'Hermann Tilke'
  'Major events'                      → 'Current: FIA WEC 8 Hours of Bahrain (2012–2017, 2019–present) Future: '
  'Webs

In [5]:
def extract_circuit_fields(soup, url):
    """
    Extract circuit metadata from a parsed Wikipedia circuit page.
    
    Returns a dict with raw strings (cleaning happens in Session 4).
    Returns None for any field that doesn't appear in the infobox.
    """
    fields_we_want = {
        "Length":    "length",
        "Turns":     "turns",
        "Opened":    "opened",
        "Architect": "architect",
        "Capacity":  "capacity",
        "Location":  "location",
    }
    
    result = {key: None for key in fields_we_want.values()}
    result["wiki_url"] = url
    
    infobox = soup.find("table", class_="infobox")
    if infobox is None:
        return result  # no infobox on this page — return Nones
    
    for row in infobox.find_all("tr"):
        label_cell = row.find("th")
        value_cell = row.find("td")
        if label_cell is None or value_cell is None:
            continue
        
        label = label_cell.get_text(" ", strip=True)
        if label in fields_we_want:
            output_key = fields_we_want[label]
            result[output_key] = value_cell.get_text(" ", strip=True)
    
    return result


# Test it on the Bahrain page we already have parsed
test_result = extract_circuit_fields(soup, url)
for key, val in test_result.items():
    print(f"  {key:<12}: {val}")

  length      : 5.417 km (3.366 mi)
  turns       : 15
  opened      : 17 March 2004 ; 22 years ago ( 2004-03-17 )
  architect   : Hermann Tilke
  capacity    : 70,000
  location    : Sakhir , Bahrain
  wiki_url    : https://en.wikipedia.org/wiki/Bahrain_International_Circuit


In [6]:
# Load the API data from Session 2 and extract one unique row per circuit
api_csv = Path("../data/raw/f1_results_2020_2025.csv")
api_df = pd.read_csv(api_csv)

# Deduplicate to get one row per circuit
circuits_df = api_df[["circuit_id", "circuit_name", "country"]].drop_duplicates().reset_index(drop=True)

# The CSV doesn't have the Wikipedia URLs — we have to reconstruct them from the raw JSON
# Load the JSON we saved, walk through it, and build a lookup: circuit_id -> wiki_url
raw_json = json.loads(Path("../data/raw/f1_results_2020_2025.json").read_text())

circuit_urls = {}
for race in raw_json:
    cid = race["Circuit"]["circuitId"]
    if cid not in circuit_urls:
        circuit_urls[cid] = race["Circuit"]["url"]

# Add wiki_url as a column by mapping circuit_id through the lookup
circuits_df["wiki_url"] = circuits_df["circuit_id"].map(circuit_urls)

# Wikipedia URLs in the API use http:// — switch to https://
circuits_df["wiki_url"] = circuits_df["wiki_url"].str.replace("http://", "https://", regex=False)

print(f"Unique circuits to scrape: {len(circuits_df)}\n")
print(circuits_df.head(10).to_string(index=False))

Unique circuits to scrape: 30

   circuit_id                         circuit_name  country                                                     wiki_url
red_bull_ring                        Red Bull Ring  Austria                  https://en.wikipedia.org/wiki/Red_Bull_Ring
  hungaroring                          Hungaroring  Hungary                    https://en.wikipedia.org/wiki/Hungaroring
  silverstone                  Silverstone Circuit       UK            https://en.wikipedia.org/wiki/Silverstone_Circuit
    catalunya       Circuit de Barcelona-Catalunya    Spain https://en.wikipedia.org/wiki/Circuit_de_Barcelona-Catalunya
          spa         Circuit de Spa-Francorchamps  Belgium   https://en.wikipedia.org/wiki/Circuit_de_Spa-Francorchamps
        monza         Autodromo Nazionale di Monza    Italy                  https://en.wikipedia.org/wiki/Monza_Circuit
      mugello Autodromo Internazionale del Mugello    Italy                https://en.wikipedia.org/wiki/Mugello_Circuit
 

In [7]:
# Scrape every circuit page, ~1.5s sleep between requests (polite, see Ethics Note)

results = []
total = len(circuits_df)

print(f"Scraping {total} circuit pages...\n")

for i, row in circuits_df.iterrows():
    cid = row["circuit_id"]
    url = row["wiki_url"]
    
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")
        
        fields = extract_circuit_fields(soup, url)
        fields["circuit_id"] = cid
        fields["circuit_name"] = row["circuit_name"]
        fields["country"] = row["country"]
        fields["http_status"] = resp.status_code
        fields["scrape_error"] = None
        
        # Quick status: did we get the key fields we care about?
        got_length = fields["length"] is not None
        got_turns = fields["turns"] is not None
        print(f"  [{i+1:2d}/{total}] {cid:<25} ✅ length={got_length} turns={got_turns}")
        
    except requests.RequestException as e:
        fields = {
            "circuit_id": cid,
            "circuit_name": row["circuit_name"],
            "country": row["country"],
            "wiki_url": url,
            "http_status": None,
            "scrape_error": str(e),
            "length": None, "turns": None, "opened": None,
            "architect": None, "capacity": None, "location": None,
        }
        print(f"  [{i+1:2d}/{total}] {cid:<25} ❌ {e}")
    
    results.append(fields)
    
    # Polite delay between requests — except after the last one
    if i < total - 1:
        time.sleep(1 + random.random())

scraped_df = pd.DataFrame(results)
print(f"\nDone. Collected {len(scraped_df)} rows.")
print(f"Columns: {list(scraped_df.columns)}")

Scraping 30 circuit pages...

  [ 1/30] red_bull_ring             ✅ length=True turns=True
  [ 2/30] hungaroring               ✅ length=True turns=True
  [ 3/30] silverstone               ✅ length=True turns=True
  [ 4/30] catalunya                 ✅ length=True turns=True
  [ 5/30] spa                       ✅ length=True turns=True
  [ 6/30] monza                     ✅ length=True turns=True
  [ 7/30] mugello                   ✅ length=True turns=True
  [ 8/30] sochi                     ✅ length=True turns=True
  [ 9/30] nurburgring               ✅ length=True turns=True
  [10/30] portimao                  ✅ length=True turns=True
  [11/30] imola                     ✅ length=True turns=True
  [12/30] istanbul                  ✅ length=True turns=True
  [13/30] bahrain                   ✅ length=True turns=True
  [14/30] yas_marina                ✅ length=True turns=True
  [15/30] monaco                    ✅ length=True turns=True
  [16/30] baku                      ✅ length=True turns

In [8]:
# Investigate why Vegas came back with length=None and turns=None

# 1. See what we DID get from Vegas
vegas_row = scraped_df[scraped_df["circuit_id"] == "vegas"].iloc[0]
print("What we extracted for Vegas:")
for col, val in vegas_row.items():
    print(f"  {col:<14}: {val}")

# 2. Re-fetch the Vegas page and look at its infobox labels directly
print("\n" + "=" * 60)
print("Infobox row labels found on the Vegas Wikipedia page:")
print("=" * 60)

vegas_url = vegas_row["wiki_url"]
resp = requests.get(vegas_url, headers=HEADERS, timeout=10)
soup_vegas = BeautifulSoup(resp.text, "lxml")
infobox = soup_vegas.find("table", class_="infobox")

if infobox is None:
    print("No infobox found at all.")
else:
    for row in infobox.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if th and td:
            label = th.get_text(" ", strip=True)
            value = td.get_text(" ", strip=True)[:60]
            print(f"  {label!r:<30} → {value!r}")

What we extracted for Vegas:
  length        : nan
  turns         : nan
  opened        : nan
  architect     : nan
  capacity      : nan
  location      : nan
  wiki_url      : https://en.wikipedia.org/wiki/Las_Vegas_Grand_Prix#Circuit
  circuit_id    : vegas
  circuit_name  : Las Vegas Strip Street Circuit
  country       : USA
  http_status   : 200
  scrape_error  : None

Infobox row labels found on the Vegas Wikipedia page:
  'Number of times held'         → '3'
  'First held'                   → '2023'
  'Most wins (drivers)'          → 'Max Verstappen (2)'
  'Most wins (constructors)'     → 'Red Bull Racing (2)'
  'Circuit length'               → '6.201 km (3.853 miles)'
  'Race length'                  → '309.958 km (192.599 miles)'
  'Laps'                         → '50'


In [9]:
# Manual fix for Vegas: the jolpica API returns the Grand Prix event URL
# (Las_Vegas_Grand_Prix#Circuit), which has a race-event infobox, not a
# circuit infobox. We override to the actual circuit page and re-scrape.
# This is the only such patch in the dataset (documented in data quality notes).

vegas_correct_url = "https://en.wikipedia.org/wiki/Las_Vegas_Strip_Circuit"

resp = requests.get(vegas_correct_url, headers=HEADERS, timeout=10)
resp.raise_for_status()
soup_vegas = BeautifulSoup(resp.text, "lxml")

vegas_fields = extract_circuit_fields(soup_vegas, vegas_correct_url)
vegas_fields["circuit_id"] = "vegas"
vegas_fields["circuit_name"] = "Las Vegas Strip Street Circuit"
vegas_fields["country"] = "USA"
vegas_fields["http_status"] = resp.status_code
vegas_fields["scrape_error"] = None

# Replace the Vegas row in scraped_df
mask = scraped_df["circuit_id"] == "vegas"
for col, val in vegas_fields.items():
    scraped_df.loc[mask, col] = val

# Verify the fix
print("Vegas row after patch:")
for col, val in scraped_df[mask].iloc[0].items():
    print(f"  {col:<14}: {val}")

Vegas row after patch:
  length        : 3.853 mi (6.201 km)
  turns         : 17
  opened        : November 16, 2023 ; 2 years ago ( 2023-11-16 )
  architect     : Carsten Tilke
  capacity      : 100,000
  location      : Paradise, Nevada , United States
  wiki_url      : https://en.wikipedia.org/wiki/Las_Vegas_Strip_Circuit
  circuit_id    : vegas
  circuit_name  : Las Vegas Strip Street Circuit
  country       : USA
  http_status   : 200
  scrape_error  : None


In [10]:
# Save scraped data with timestamp — never overwrite raw data
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = DATA_RAW / f"circuits_wiki_{ts}.csv"
parquet_path = DATA_RAW / f"circuits_wiki_{ts}.parquet"

scraped_df.to_csv(csv_path, index=False)
scraped_df.to_parquet(parquet_path, index=False)

print(f"Saved {len(scraped_df)} rows × {len(scraped_df.columns)} columns to:")
print(f"  {csv_path}  ({csv_path.stat().st_size:,} bytes)")
print(f"  {parquet_path}  ({parquet_path.stat().st_size:,} bytes)")

Saved 30 rows × 12 columns to:
  ../data/raw/circuits_wiki_20260524_182752.csv  (7,013 bytes)
  ../data/raw/circuits_wiki_20260524_182752.parquet  (12,402 bytes)


## Data Dictionary

Scraped output saved to `data/raw/circuits_wiki_<timestamp>.csv` and `.parquet`. One row per F1 circuit appearing in the 2020–2025 seasons.

| Column         | Type      | Source                  | Description                                                                 |
|----------------|-----------|-------------------------|-----------------------------------------------------------------------------|
| `circuit_id`   | string    | jolpica API (carried)   | Ergast-style identifier, e.g. `bahrain`, `monza`. Join key with API data.   |
| `circuit_name` | string    | jolpica API (carried)   | Full circuit name, e.g. "Autodromo Nazionale di Monza".                     |
| `country`      | string    | jolpica API (carried)   | Country, e.g. "Italy".                                                      |
| `wiki_url`     | string    | jolpica API + override  | Wikipedia URL for the circuit page. Manually overridden for Vegas.          |
| `length`       | string    | Wikipedia infobox       | Raw track length, e.g. "5.412 km (3.363 mi)". Needs parsing in Session 4.   |
| `turns`        | string    | Wikipedia infobox       | Raw number of corners, e.g. "15" or "15 (5 layouts)". Needs parsing.        |
| `opened`       | string    | Wikipedia infobox       | Date opened, e.g. "17 March 2004". Needs date parsing.                      |
| `architect`    | string    | Wikipedia infobox       | Track designer, e.g. "Hermann Tilke". Free text.                            |
| `capacity`     | string    | Wikipedia infobox       | Spectator capacity, e.g. "70,000". Comma thousands separator.               |
| `location`     | string    | Wikipedia infobox       | City/region, e.g. "Sakhir, Bahrain". Free text.                             |
| `http_status`  | int       | requests response       | HTTP status of the fetch (200 = OK). Diagnostic.                            |
| `scrape_error` | string    | exception handler       | Error message if the fetch failed, else `None`. Diagnostic.                 |

## Data Quality Notes

Known issues in the raw scraped data, to be handled in Session 4 (`03_cleaning.ipynb`):

1. **Length and turns are raw strings, not numbers.** `length` contains both km and mi units in one string (e.g. `"5.412 km (3.363 mi)"`), often with non-breaking spaces (`\xa0`). Will extract the km value and cast to float. `turns` sometimes includes layout annotations (e.g. `"15 (5 layouts)"`); will keep only the primary digit.

2. **Dates need parsing.** The `opened` field is free-text like `"17 March 2004"` or `"November 16, 2023"`, sometimes appended with `"; 2 years ago"` or HTML artifacts. Will extract the year only and cast to int, since we only need it for circuit-age features.

3. **Vegas required a manual URL override.** The jolpica API returned `Las_Vegas_Grand_Prix#Circuit` (the race event page, which has an event-style infobox) instead of `Las_Vegas_Strip_Circuit` (the actual track page). We documented and patched this in the notebook. This is the only such patch; all 29 other circuits used the standard URL convention. Worth re-checking in future seasons if new circuits are added.

4. **30 rows is below the brief's 200-row threshold for the scraped table.** Justification: the project's primary dataset (jolpica API, 2,618 rows) already exceeds 200. Wikipedia is enrichment data — one row per *unique* circuit — and there are simply 30 distinct circuits in 2020–2025. The Wikipedia scrape multiplies value by joining onto the 2,618-row API table downstream, not by being large itself.